# Phase II_01: Spectral Relabeling with USGS MTBS Standards

**Objective**: Create multi-class burn labels using official USGS burn severity classifications

**Approach**: Classify each sample using delta Normalized Burn Ratio (dNBR) with USGS MTBS thresholds

**Input**: CaBuAr dataset with pre-fire and post-fire Sentinel-2 images

**Output**: Multi-class labels (7 classes) based on dNBR change

**Reference**: USGS MTBS standards for dNBR classification

## Setup: Google Drive and Dependencies

In [ ]:
import sys
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
import json

sys.path.insert(0, '/content/RETINNA/src')

# Mount Google Drive
print("📁 Initializing Google Drive...")
from drive_utils import setup_drive_for_colab

drive_manager = setup_drive_for_colab(verbose=True)

if drive_manager is None:
    raise RuntimeError("Failed to connect to Google Drive - aborting")

print("\n" + "="*70)
print("PHASE II_01: SPECTRAL RELABELING WITH USGS MTBS STANDARDS")
print("="*70)
print(f"Session start: {datetime.now().isoformat()}")
print(f"Drive cache: {drive_manager.root}")

## Load Dataset

In [ ]:
%pip install -q torchgeo torch torchvision matplotlib numpy
from dataset import get_dataloaders
from colab_utils import setup_cabuaur_cached

# Setup Google Drive caching if on Colab
root = None
try:
    root = setup_cabuaur_cached()
    print("✓ Google Drive caching enabled")
except (ImportError, RuntimeError) as e:
    print(f"⚠ Drive caching failed: {e}")
    print("Using default TorchGeo cache")
    root = '/tmp/cabuaur'

# Load dataloaders (all splits)
print("\n📊 Loading CaBuAr dataset...")
dataloaders = get_dataloaders(batch_size=1, num_workers=0, normalize=True, root=root)

train_dataset = dataloaders['datasets']['train']
val_dataset = dataloaders['datasets']['val']
test_dataset = dataloaders['datasets']['test']

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Val samples: {len(val_dataset)}")
print(f"✓ Test samples: {len(test_dataset)}")
print(f"✓ Total: {len(train_dataset) + len(val_dataset) + len(test_dataset)} samples")

## Spectral Analysis: Compute USGS NBR and Classification

In [ ]:
def compute_spectral_indices(pre_fire, post_fire):
    """
    Compute spectral indices for burn classification.
    
    Uses RdNBR (Relativized dNBR) which is better for arid/sparse vegetation regions.
    RdNBR = dNBR / sqrt(|NBRpre|) normalizes by pre-fire vegetation amount.
    
    Args:
        pre_fire: [12, H, W] pre-fire Sentinel-2 bands
        post_fire: [12, H, W] post-fire Sentinel-2 bands
    
    Returns:
        dict with NBR values, dNBR, RdNBR, and indices
    """
    # Standard burn detection bands: B08 (NIR) and B12 (SWIR)
    nir_idx, swir_idx, green_idx, blue_idx = 7, 11, 2, 1
    
    # Extract bands
    nir_pre = pre_fire[nir_idx]
    nir_post = post_fire[nir_idx]
    swir_pre = pre_fire[swir_idx]
    swir_post = post_fire[swir_idx]
    green_post = post_fire[green_idx]
    blue_post = post_fire[blue_idx]
    
    eps = 1e-8
    
    # NBR = (NIR - SWIR) / (NIR + SWIR)
    nbr_pre = (nir_pre - swir_pre) / (nir_pre + swir_pre + eps)
    nbr_post = (nir_post - swir_post) / (nir_post + swir_post + eps)
    
    # dNBR = NBRpre - NBRpost (positive = burned)
    dnbr = nbr_pre - nbr_post
    
    # RdNBR = dNBR / sqrt(|NBRpre|) - relativized for sparse vegetation
    rdnbr = dnbr / (np.sqrt(np.abs(nbr_pre)) + eps)
    
    # MNDWI = (Green - SWIR) / (Green + SWIR) for water detection
    mndwi = (green_post - swir_post) / (green_post + swir_post + eps)
    
    return {
        'nbr_pre': nbr_pre,
        'nbr_post': nbr_post,
        'dnbr': dnbr,
        'rdnbr': rdnbr,
        'mndwi': mndwi,
        'blue_post': blue_post,
        'nir_post': nir_post
    }


def classify_by_rdnbr(rdnbr, mndwi, blue, nir):
    """
    Classify pixels using RdNBR (Relativized dNBR) with USGS MTBS thresholds.
    
    RdNBR is better for arid/sparse regions like Southern California.
    Returns class labels (0-6) based on RdNBR change normalized by pre-fire vegetation.
    This establishes ground truth for threshold calibration.
    """
    labels = torch.zeros_like(rdnbr, dtype=torch.uint8)
    
    # Cloud detection
    cloud_mask = (blue > 0.25) & ((blue / (nir + 1e-8)) > 0.8)
    labels[cloud_mask] = 6
    
    # Water detection
    water_mask = (mndwi > 0.3) & ~cloud_mask
    labels[water_mask] = 5
    
    non_special = ~cloud_mask & ~water_mask
    
    # USGS MTBS thresholds applied to RdNBR
    labels[(rdnbr >= 0.66) & non_special] = 4     # Extreme
    labels[(rdnbr >= 0.44) & (rdnbr < 0.66) & non_special] = 3   # High
    labels[(rdnbr >= 0.27) & (rdnbr < 0.44) & non_special] = 2   # Moderate
    labels[(rdnbr >= 0.1) & (rdnbr < 0.27) & non_special] = 1    # Low
    # Unburned (< 0.1): already 0
    
    return labels


def calibrate_nbr_thresholds(nbr_post, rdnbr_labels):
    """
    Establish NBR thresholds by analyzing post-fire NBR values for each RdNBR class.
    
    Returns dict of threshold boundaries based on observed NBR distributions.
    """
    thresholds = {}
    
    for class_id in range(5):  # 0-4 (ignoring water/cloud)
        mask = (rdnbr_labels == class_id)
        if mask.sum() > 0:
            class_nbr = nbr_post[mask]
            thresholds[class_id] = {
                'min': float(class_nbr.min()),
                'max': float(class_nbr.max()),
                'mean': float(class_nbr.mean())
            }
    
    return thresholds


def classify_by_nbr(nbr, rdnbr_labels, mndwi, blue, nir):
    """
    Classify pixels using single-image NBR, using thresholds calibrated from RdNBR.
    
    Cloud/water detection same as RdNBR; severity classes derived from NBR ranges
    observed in post-fire image.
    """
    labels = torch.zeros_like(nbr, dtype=torch.uint8)
    
    # Cloud and water detection (same as RdNBR)
    cloud_mask = (blue > 0.25) & ((blue / (nir + 1e-8)) > 0.8)
    labels[cloud_mask] = 6
    
    water_mask = (mndwi > 0.3) & ~cloud_mask
    labels[water_mask] = 5
    
    non_special = ~cloud_mask & ~water_mask
    
    # For severity classes, use percentile-based thresholds derived from RdNBR distribution
    # Get NBR values for each severity class (from post-fire, where RdNBR categories are known)
    for class_id in [4, 3, 2, 1]:  # Extreme, High, Moderate, Low
        mask_class = (rdnbr_labels == class_id) & non_special
        if mask_class.sum() > 0:
            class_nbr_values = nbr[mask_class]
            threshold_min = class_nbr_values.min()
            
            # Assign this class to pixels with NBR in this range
            if class_id == 4:  # Extreme
                labels[(nbr >= threshold_min) & non_special] = 4
            elif class_id == 3:  # High
                labels[(nbr >= threshold_min) & (nbr < class_nbr_values.max()) & non_special] = 3
            elif class_id == 2:  # Moderate
                labels[(nbr >= threshold_min) & (nbr < class_nbr_values.max()) & non_special] = 2
            elif class_id == 1:  # Low
                labels[(nbr >= threshold_min) & (nbr < class_nbr_values.max()) & non_special] = 1
    
    # Remaining pixels (lowest NBR) are Unburned
    return labels


print("✓ Spectral analysis functions defined (RdNBR calibration → NBR classification)")

## Process All Samples: Train + Val + Test

In [ ]:
def process_dataset(dataset, dataset_name):
    """
    Process dataset: use RdNBR to calibrate thresholds, then classify pre and post images.
    Also compute and save difference images for Phase II_02 change-detection training.

    RdNBR (Relativized dNBR) is better for arid/sparse regions like California.

    For each sample:
    1. Compute RdNBR and classify using USGS thresholds (ground truth)
    2. Calibrate single-image NBR thresholds from RdNBR classes
    3. Apply those thresholds to classify pre and post images independently
    4. Compute (Post - Pre) difference images for NAIP-compatible bands (RGBN)
    """
    print(f"\n📊 Processing {dataset_name} set ({len(dataset)} samples)...")

    all_labels_pre = []
    all_labels_post = []
    all_diffs = []  # Store (Post - Pre) difference images

    class_counts_pre = {i: 0 for i in range(7)}
    class_counts_post = {i: 0 for i in range(7)}

    # Band indices for NAIP-compatible channels
    # CaBuAr band order: B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12, CLP, SCL
    # (0,    1,    2,    3,    4,    5,    6,    7,    8,    9,    10,  11)
    band_indices = {
        'R': 2,      # B04 (Red)
        'G': 1,      # B03 (Green)
        'B': 0,      # B02 (Blue)
        'NIR': 6,    # B08 (NIR)
    }
    rgbn_indices = [band_indices['R'], band_indices['G'], band_indices['B'], band_indices['NIR']]

    for sample_idx in range(len(dataset)):
        if (sample_idx + 1) % max(1, len(dataset) // 10) == 0:
            print(f"  {sample_idx + 1}/{len(dataset)} samples processed...")

        sample = dataset[sample_idx]
        image = sample['image'].numpy()  # [2, 12, 512, 512]

        pre_fire = image[0]
        post_fire = image[1]

        # Step 1: Compute spectral indices
        indices = compute_spectral_indices(pre_fire, post_fire)
        rdnbr = torch.from_numpy(indices['rdnbr']).float()
        nbr_pre = torch.from_numpy(indices['nbr_pre']).float()
        nbr_post = torch.from_numpy(indices['nbr_post']).float()
        mndwi = torch.from_numpy(indices['mndwi']).float()
        blue_post = torch.from_numpy(indices['blue_post']).float()
        nir_post = torch.from_numpy(indices['nir_post']).float()

        # Step 2: Classify by RdNBR (establishes ground truth)
        labels_rdnbr = classify_by_rdnbr(rdnbr, mndwi, blue_post, nir_post)

        # Step 3: Calibrate NBR thresholds from RdNBR classification
        thresholds = calibrate_nbr_thresholds(nbr_post, labels_rdnbr)

        # Step 4: Classify pre-fire using calibrated thresholds
        labels_pre = classify_by_nbr(nbr_pre, labels_rdnbr, mndwi, blue_post, nir_post)
        labels_pre = labels_pre.numpy()

        # Step 5: Classify post-fire using calibrated thresholds
        labels_post = classify_by_nbr(nbr_post, labels_rdnbr, mndwi, blue_post, nir_post)
        labels_post = labels_post.numpy()

        # Step 6: Compute (Post - Pre) difference for NAIP-compatible bands
        # Extract RGBN for both pre and post
        pre_rgbn = pre_fire[rgbn_indices]  # [4, 512, 512]
        post_rgbn = post_fire[rgbn_indices]  # [4, 512, 512]
        diff_rgbn = post_rgbn - pre_rgbn  # [4, 512, 512] change detection input
        all_diffs.append(diff_rgbn)

        # Store labels
        all_labels_pre.append(labels_pre)
        all_labels_post.append(labels_post)

        # Track class distributions
        for class_id in range(7):
            class_counts_pre[class_id] += (labels_pre == class_id).sum()
            class_counts_post[class_id] += (labels_post == class_id).sum()
    
    # Stack labels
    labels_pre_array = np.stack(all_labels_pre, axis=0)  # [N, 512, 512]
    labels_post_array = np.stack(all_labels_post, axis=0)  # [N, 512, 512]
    diffs_array = np.stack(all_diffs, axis=0)  # [N, 4, 512, 512]
    
    # Compute metrics
    total_pixels = labels_pre_array.size
    metrics = {
        'dataset': dataset_name,
        'n_samples': len(dataset),
        'total_pixels': int(total_pixels),
        'pre_fire': {'class_distribution': {}},
        'post_fire': {'class_distribution': {}}
    }
    
    class_names = [
        'Unburned',
        'Low Severity',
        'Moderate Severity',
        'High Severity',
        'Extreme Severity',
        'Water',
        'Cloud/Shadow'
    ]
    
    print(f"\n  Pre-fire distribution:")
    for class_id in range(7):
        count = int(class_counts_pre[class_id])
        pct = 100.0 * count / total_pixels
        metrics['pre_fire']['class_distribution'][class_names[class_id]] = {
            'pixels': count,
            'percentage': float(f"{pct:.2f}")
        }
        print(f"    Class {class_id} ({class_names[class_id]:20}): {count:12,} px ({pct:5.2f}%)")
    
    print(f"\n  Post-fire distribution:")
    for class_id in range(7):
        count = int(class_counts_post[class_id])
        pct = 100.0 * count / total_pixels
        metrics['post_fire']['class_distribution'][class_names[class_id]] = {
            'pixels': count,
            'percentage': float(f"{pct:.2f}")
        }
        print(f"    Class {class_id} ({class_names[class_id]:20}): {count:12,} px ({pct:5.2f}%)")
    
    return labels_pre_array, labels_post_array, diffs_array, metrics


# Process all splits
train_pre, train_post, train_diffs, train_metrics = process_dataset(train_dataset, "Train")
val_pre, val_post, val_diffs, val_metrics = process_dataset(val_dataset, "Val")
test_pre, test_post, test_diffs, test_metrics = process_dataset(test_dataset, "Test")

print("\n✓ All datasets processed")

## Combine and Save Results

In [ ]:
# Combine labels for training (doubles training data)
all_labels = np.concatenate([
    train_pre, train_post,
    val_pre, val_post,
    test_pre, test_post
], axis=0)

# Combine difference images for Phase II_02 training
all_diffs = np.concatenate([
    train_diffs, val_diffs, test_diffs
], axis=0)

all_metrics = {
    'timestamp': datetime.now().isoformat(),
    'note': 'Labels: RdNBR establishes thresholds (better for arid regions), then applied to pre and post images independently',
    'method': 'RdNBR (Relativized dNBR) = dNBR / sqrt(|NBRpre|) - accounts for pre-fire vegetation amount',
    'total_samples': len(train_dataset) + len(val_dataset) + len(test_dataset),
    'total_images': 2 * (len(train_dataset) + len(val_dataset) + len(test_dataset)),
    'phase_ii_02_notes': 'Difference images (Post - Pre) for 4 NAIP-compatible bands saved separately for change-detection training',
    'splits': {
        'train': train_metrics,
        'val': val_metrics,
        'test': test_metrics
    }
}

print(f"\n💾 Saving results...")
print(f"  Labels shape: {all_labels.shape} (pre + post combined)")
print(f"  Difference images shape: {all_diffs.shape} (train + val + test combined)")
print(f"  Total images for training: {all_metrics['total_images']}")
print(f"  Method: {all_metrics['method']}")

# Save multi-class labels to Drive
labels_path = drive_manager.save_with_timestamp(
    torch.from_numpy(all_labels),
    rel_path="phase2/II_01_relabeling",
    filename_base="multi_class_labels",
    file_format=".pt",
    verbose=True
)

# Save difference images (RGBN) for Phase II_02 to Drive
diffs_path = drive_manager.save_with_timestamp(
    torch.from_numpy(all_diffs).float(),
    rel_path="phase2/II_01_relabeling",
    filename_base="difference_images_rgbn",
    file_format=".pt",
    verbose=True
)

# Save metrics to Drive
metrics_path = drive_manager.save_with_timestamp(
    all_metrics,
    rel_path="phase2/II_01_relabeling",
    filename_base="metrics",
    file_format=".json",
    verbose=True
)

if labels_path and diffs_path and metrics_path:
    print(f"\n✓ Results saved to Drive")
    print(f"  Labels: {labels_path.name}")
    print(f"  Difference images (Phase II_02): {diffs_path.name}")
    print(f"  Metrics: {metrics_path.name}")
else:
    print(f"\n✗ Failed to save to Drive")

## Quality Control: Verify Results

In [ ]:
# Reload from Drive to verify
print("\n🔍 Quality Control: Verifying saved files...")

latest_labels = drive_manager.get_latest_file(
    rel_path="phase2/II_01_relabeling",
    filename_pattern="multi_class_labels"
)

latest_diffs = drive_manager.get_latest_file(
    rel_path="phase2/II_01_relabeling",
    filename_pattern="difference_images_rgbn"
)

if latest_labels and latest_diffs:
    loaded_labels = torch.load(latest_labels)
    loaded_diffs = torch.load(latest_diffs)
    print(f"✓ Labels verified: {loaded_labels.shape}")
    print(f"✓ Difference images verified: {loaded_diffs.shape}")
    
    # Check class distribution
    print(f"\nClass distribution in loaded labels:")
    class_names = [
        'Unburned',
        'Low Severity',
        'Moderate Severity',
        'High Severity',
        'Extreme Severity',
        'Water',
        'Cloud/Shadow'
    ]
    
    for class_id in range(7):
        count = (loaded_labels == class_id).sum().item()
        total = loaded_labels.numel()
        pct = 100.0 * count / total
        print(f"  {class_id}: {class_names[class_id]:20} {count:12,} px ({pct:5.2f}%)")
    
    print(f"\nDifference image statistics:")
    print(f"  Min: {loaded_diffs.min():.4f}")
    print(f"  Max: {loaded_diffs.max():.4f}")
    print(f"  Mean: {loaded_diffs.mean():.4f}")
    print(f"  Std: {loaded_diffs.std():.4f}")
else:
    print(f"✗ Could not verify saved files")

print(f"\n✓ Phase II_01 complete at {datetime.now().isoformat()}")
print(f"\n📌 Output files for Phase II_02:")
print(f"  - difference_images_rgbn_*.pt: Ready for change-detection training")
print(f"  - multi_class_labels_*.pt: Labels for training targets")
print(f"\nNext: II_02_unet_training.ipynb (change-detection U-Net)")